# Task 06: 1D diffusion model

This notebook is built step by step. The goal is to train a very small diffusion model that learns a one-dimensional mixture of two Gaussians centered at $-4$ and $4$.

A diffusion model has two processes:

- the **forward process** slowly adds Gaussian noise to real data until it looks almost like standard normal noise,
- the **reverse process** starts from noise and uses a neural network to remove the noise step by step.

Compared with a GAN, there is no discriminator. Compared with a VAE, we do not directly learn an encoder-decoder pair. Here we train a network to predict the noise that was added at a randomly chosen diffusion timestep.

### 1. Imports and setup

This first cell configures plotting first, imports PyTorch tools, sets seeds, and selects the best available device.

In [ ]:
from pathlib import Path

from common.setup_plotting import setup_matplotlib, get_figure_dir

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from tqdm import tqdm

setup_matplotlib()        # configure matplotlib first (So i can use LaTeX in the labels)
# Make interactive plots work in Jupyter notebooks 
%matplotlib inline  
import matplotlib.pyplot as plt   # THEN import pyplot

import seaborn as sns

torch.manual_seed(42)
np.random.seed(42)

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

print("device:", device)

In [ ]:
# get and if needed create figure directory for this task
fig_dir = get_figure_dir("task_06")

# get and if needed create data directory for this task
# The data is synthetic, but if you want to save it, you can do so in this directory.
data_dir = Path("../../../data/task_06")
data_dir.mkdir(parents=True, exist_ok=True)

### 2. Dataset generation and visualization

The target distribution is a balanced mixture of two one-dimensional Gaussian distributions:

$p(x) = \frac{1}{2}\mathcal{N}(-4, 1) + \frac{1}{2}\mathcal{N}(4, 1).$

We can sample from this distribution directly. These samples are the "real data" that the diffusion model should learn to generate later.

In [ ]:
def sample_two_gaussians(num_samples, left_mode_probability=0.5):
    """Sample a 1D Gaussian mixture with controllable mode probabilities."""
    component = torch.rand(num_samples, 1)
    means = torch.where(component < left_mode_probability, -4.0, 4.0)
    noise = torch.randn(num_samples, 1)
    return means + noise


left_mode_probability = 0.25
num_train_samples = 10_000
num_validation_samples = 2_000

train_samples = sample_two_gaussians(num_train_samples, left_mode_probability)
validation_samples = sample_two_gaussians(num_validation_samples, left_mode_probability)

train_dataset = TensorDataset(train_samples)
train_loader = DataLoader(
    train_dataset,
    batch_size=256,
    shuffle=True,
)

print("left mode probability:", left_mode_probability)
print("right mode probability:", 1.0 - left_mode_probability)
print("train_samples shape:", train_samples.shape)
print("validation_samples shape:", validation_samples.shape)
print("number of training batches:", len(train_loader))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
bins = np.linspace(-9, 9, 70)

sns.histplot(train_samples.squeeze().numpy(), bins=bins, stat="density", ax=ax, color="tab:blue", alpha=0.35, label="training samples")
sns.kdeplot(train_samples.squeeze().numpy(), ax=ax, color="tab:blue", linewidth=2, label="estimated density")

ax.set_xlabel(r"$x_0$")
ax.set_ylabel("density")
ax.set_title("Target distribution: two Gaussian modes")
ax.legend()
fig.tight_layout()
fig.savefig(fig_dir / "target_distribution.pdf")

### 3. Diffusion hyperparameters

We use the fixed value $\beta = 0.02$ requested in the task. At each forward step, the current sample keeps a factor of $\sqrt{1 - \beta}$ and receives a small amount of new Gaussian noise.

It is useful to precompute

\[
\alpha_t = 1 - \beta_t, \qquad \bar{\alpha}_t = \prod_{s=1}^{t} \alpha_s.
\]

Then we can sample any noisy version $x_t$ directly from clean data $x_0$:

\[
x_t = \sqrt{\bar{\alpha}_t}x_0 + \sqrt{1 - \bar{\alpha}_t}\epsilon,
\]

where $\epsilon \sim \mathcal{N}(0, 1)$.

In [ ]:
time_steps = 250
beta = 0.02
num_epochs = 150
learning_rate = 8e-4

betas = torch.full((time_steps,), beta, device=device)
alphas = 1.0 - betas
alpha_bars = torch.cumprod(alphas, dim=0)

print("time_steps:", time_steps)
print("beta:", beta)
print("alpha_bar at final step:", alpha_bars[-1].item())
print("num_epochs:", num_epochs)
print("learning_rate:", learning_rate)

### 4. Forward diffusion process

The forward process is not learned. It is a fixed corruption process that turns real data into noise. We use it in two ways:

- during training, to create noisy inputs $x_t$ and known target noise $\epsilon$,
- for visualization, to show how the two clear modes disappear as $t$ increases.

In [ ]:
def extract(values, timesteps, x_shape):
    """Gather one scalar schedule value per batch item and reshape it for broadcasting."""
    batch_size = timesteps.shape[0]
    gathered = values.gather(0, timesteps)

    # Add singleton dimensions after the batch dimension so the scalar schedule values broadcast to x0.
    return gathered.reshape(batch_size, *((1,) * (len(x_shape) - 1)))

# With fixed beta, chaining t Gaussian noise steps equals one direct jump using alpha_bar = alpha**t.
# the accumulated noise variance is the geometric sum 1 - alpha_bar, so the noise is scaled by sqrt(1 - alpha_bar).
# xt = sqrt_alpha_bar * x0 + sqrt_one_minus_alpha_bar * noise
def q_sample(x0, timesteps, noise=None):
    """Sample x_t from x_0 at the requested timesteps."""
    if noise is None:
        noise = torch.randn_like(x0)

    alpha_bar = extract(alpha_bars, timesteps, x0.shape)
    sqrt_alpha_bar = torch.sqrt(alpha_bar)
    xt = sqrt_alpha_bar * x0 + torch.sqrt(1.0 - alpha_bar) * noise
    return xt, noise

In [ ]:
forward_steps_to_plot = [0, 25, 75, 150, 249]
plot_count = 4_000
x0_plot = train_samples[:plot_count].to(device)

plot_range = (-8, 8)
fig, axes = plt.subplots(2, 3, figsize=(8.2, 4.8), sharex=True, sharey=True)
axes = axes.ravel()
bins = np.linspace(*plot_range, 60)

for ax, step in zip(axes, forward_steps_to_plot):
    timesteps = torch.full((plot_count,), step, device=device, dtype=torch.long)
    xt, _ = q_sample(x0_plot, timesteps)
    ax.hist(xt.cpu().squeeze().numpy(), bins=bins, density=True, color="tab:orange", alpha=0.75)
    ax.set_xlim(plot_range)
    ax.set_title(f"t = {step}", fontsize=12)
    ax.tick_params(labelsize=10)

axes[-1].axis("off")
for ax in axes[3:5]:
    ax.set_xlabel(r"$x_t$", fontsize=11)

axes[0].set_ylabel("density", fontsize=11)
axes[3].set_ylabel("density", fontsize=11)
fig.suptitle("Forward diffusion: the target distribution is gradually noised", fontsize=13)
fig.tight_layout(rect=(0, 0, 1, 0.94))
fig.savefig(fig_dir / "forward_diffusion_steps.pdf")

### 5. Noise-predicting MLP

The neural network receives two numbers:

- the noisy scalar value $x_t$,
- the normalized timestep $t / T$.

It outputs one number: the predicted noise $\hat{\epsilon}_	heta(x_t, t)$. A small MLP is enough because the data is only one-dimensional.

In [ ]:
class NoisePredictor(nn.Module):
    def __init__(self, hidden_units=64):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(2, hidden_units),
            nn.SiLU(),
            nn.Linear(hidden_units, hidden_units),
            nn.SiLU(),
            nn.Linear(hidden_units, hidden_units),
            nn.SiLU(),
            nn.Linear(hidden_units, 1),
        )

    def forward(self, xt, timesteps):
        normalized_t = timesteps.float().unsqueeze(1) / (time_steps - 1)
        model_input = torch.cat([xt, normalized_t], dim=1)
        return self.network(model_input)


noise_predictor = NoisePredictor().to(device)
optimizer = torch.optim.Adam(noise_predictor.parameters(), lr=learning_rate)
loss_function = nn.MSELoss()

print(noise_predictor)
print("number of trainable parameters:", sum(p.numel() for p in noise_predictor.parameters() if p.requires_grad))

### 6. Training loop

Training follows the core idea from DDPM Algorithm 1:

1. sample clean data $x_0$,
2. sample a random timestep $t$,
3. sample Gaussian noise $\epsilon$,
4. construct $x_t$ using the forward process,
5. train the network to predict $\epsilon$ from $(x_t, t)$.

The loss is mean squared error between the true noise and predicted noise.

In [ ]:
def validation_loss(model, validation_data):
    model.eval()
    with torch.no_grad():
        x0 = validation_data.to(device)
        timesteps = torch.randint(0, time_steps, (x0.shape[0],), device=device)
        xt, true_noise = q_sample(x0, timesteps)
        predicted_noise = model(xt, timesteps)
        loss = loss_function(predicted_noise, true_noise)
    return loss.item()


training_losses = []
validation_losses = []
progress_bar = tqdm(range(num_epochs))

for epoch in progress_bar:
    noise_predictor.train()
    epoch_losses = []

    for (x0,) in train_loader:
        x0 = x0.to(device)
        timesteps = torch.randint(0, time_steps, (x0.shape[0],), device=device)
        xt, true_noise = q_sample(x0, timesteps)

        predicted_noise = noise_predictor(xt, timesteps)
        loss = loss_function(predicted_noise, true_noise)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_losses.append(loss.item())

    train_loss = float(np.mean(epoch_losses))
    val_loss = validation_loss(noise_predictor, validation_samples)
    training_losses.append(train_loss)
    validation_losses.append(val_loss)

    progress_bar.set_description(f"train {train_loss:.4f} | val {val_loss:.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(training_losses, label="training loss")
ax.plot(validation_losses, label="validation loss")
ax.set_xlabel("epoch")
ax.set_ylabel("MSE noise prediction loss")
ax.set_title("Noise prediction training curve")
ax.legend()
fig.tight_layout()
fig.savefig(fig_dir / "noise_prediction_loss.pdf")

print("final training loss:", training_losses[-1])
print("final validation loss:", validation_losses[-1])

### 7. Reverse diffusion sampler

The reverse process starts with pure Gaussian noise $x_T$ and repeatedly applies the learned denoising model. For this fixed-variance DDPM sampler, one reverse step uses

\[
\mu_\theta(x_t, t) = \frac{1}{\sqrt{\alpha_t}}\left(x_t - \frac{\beta_t}{\sqrt{1 - \bar{\alpha}_t}}\hat{\epsilon}_\theta(x_t, t)\right).
\]

For every step except the final one, we add fresh Gaussian noise scaled by $\sqrt{\beta_t}$. The final output should look like the original two-Gaussian target distribution.

In [ ]:
@torch.no_grad()
def sample_reverse(model, count, history_steps=None):
    """Generate samples by running the learned reverse diffusion chain."""
    model.eval()
    x = torch.randn(count, 1, device=device)

    if history_steps is None:
        history_steps = []
    history_steps = set(history_steps)
    history = {}

    if time_steps in history_steps:
        history[time_steps] = x.detach().cpu()

    for step in reversed(range(time_steps)):
        timesteps = torch.full((count,), step, device=device, dtype=torch.long)
        predicted_noise = model(x, timesteps)

        alpha_t = alphas[step]
        beta_t = betas[step]
        alpha_bar_t = alpha_bars[step]

        mean = (1.0 / torch.sqrt(alpha_t)) * (
            x - (beta_t / torch.sqrt(1.0 - alpha_bar_t)) * predicted_noise
        )

        if step > 0:
            z = torch.randn_like(x)
            x = mean + torch.sqrt(beta_t) * z
        else:
            x = mean

        if step in history_steps:
            history[step] = x.detach().cpu()

    if history_steps:
        return x.detach().cpu(), history
    return x.detach().cpu()

### 8. Visualize reverse diffusion steps

The next plot shows snapshots of the reverse process. At $t = 250$ the samples are standard Gaussian noise. As the learned denoising chain runs toward $t = 0$, the samples should split into the two modes near $-4$ and $4$.

In [ ]:
reverse_steps_to_plot = [250, 200, 150, 75, 25, 0]
generated_samples, reverse_history = sample_reverse(
    noise_predictor,
    count=4_000,
    history_steps=reverse_steps_to_plot,
)

plot_range = (-8, 8)
fig, axes = plt.subplots(2, 3, figsize=(8.2, 4.8), sharex=True, sharey=True)
axes = axes.ravel()
bins = np.linspace(*plot_range, 60)

for ax, step in zip(axes, reverse_steps_to_plot):
    values = reverse_history[step].squeeze().numpy()
    ax.hist(values, bins=bins, density=True, color="tab:green", alpha=0.75)
    ax.set_xlim(plot_range)
    ax.set_title(f"t = {step}", fontsize=12)
    ax.tick_params(labelsize=10)

for ax in axes[3:]:
    ax.set_xlabel(r"$x_t$", fontsize=11)

axes[0].set_ylabel("density", fontsize=11)
axes[3].set_ylabel("density", fontsize=11)
fig.suptitle("Reverse diffusion: noise is transformed into generated samples", fontsize=13)
fig.tight_layout(rect=(0, 0, 1, 0.94))
fig.savefig(fig_dir / "reverse_diffusion_steps.pdf")

### 9. Final comparison

Finally we compare generated samples with fresh samples from the true target distribution. A successful minimal model should recover both modes and roughly the right spread around each mode.

In [ ]:
final_generated = sample_reverse(noise_predictor, count=10_000).squeeze().numpy()
true_comparison = sample_two_gaussians(10_000, left_mode_probability).squeeze().numpy()

plot_range = (-8, 8)
fig, ax = plt.subplots(figsize=(4.8, 3.4))
bins = np.linspace(*plot_range, 70)

ax.hist(true_comparison, bins=bins, density=True, alpha=0.35, color="tab:blue", label="true distribution")
ax.hist(final_generated, bins=bins, density=True, alpha=0.35, color="tab:red", label="generated distribution")

sns.kdeplot(true_comparison, ax=ax, color="tab:blue", linewidth=2.5)
sns.kdeplot(final_generated, ax=ax, color="tab:red", linewidth=2.5)

ax.set_xlim(plot_range)
ax.set_xlabel(r"$x$", fontsize=11)
ax.set_ylabel("density", fontsize=11)
ax.set_title("Final generated samples vs. target samples", fontsize=12)
ax.tick_params(labelsize=10)
ax.legend(fontsize=9)
fig.tight_layout()
fig.savefig(fig_dir / "final_generated_vs_target.pdf")

print("true mean:", true_comparison.mean())
print("generated mean:", final_generated.mean())
print("true std:", true_comparison.std())
print("generated std:", final_generated.std())

### 10. Conclusion

This notebook implements the smallest useful version of a DDPM-style diffusion model:

- the data is a one-dimensional two-Gaussian mixture,
- the forward process is fixed and gradually adds Gaussian noise,
- the MLP learns to predict the noise in $x_t$ from the scalar value and timestep,
- the reverse sampler starts from Gaussian noise and uses the learned noise predictions to generate new samples.

This samplers purpose is to make the mechanics of diffusion visible: noise is added by a known process, and generation works by learning how to undo that process one step at a time.